# 1 lab1 openai agent with custom model: practice

Configuration stays outside executable examples → Provider endpoint and model selection → Configure a real SDK agent and inspect its tool schema → Client, request, and response are separate objects → Await vs. concurrent work

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/community_contributions/1_lab1_openai_agent_with_custom_model.ipynb](../../../../../../../2_openai/community_contributions/1_lab1_openai_agent_with_custom_model.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. Provider endpoint and model selection

A compatible client can target different services. The endpoint selects the provider and its credentials; the model identifier selects a model there. Compatibility does not guarantee every feature works.

**Predict:** Does changing only the variable name change the provider?

In [ ]:
# Non-secret connection descriptions only; no clients or keys are created.
providers = {
    "openai": {"key_variable": "OPENAI_API_KEY", "endpoint": "default"},
    "gemini": {"key_variable": "GOOGLE_API_KEY", "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai/"},
    "ollama": {"key_variable": None, "endpoint": "http://localhost:11434/v1"},
}
for provider, settings in providers.items():
    print(provider, settings["endpoint"], settings["key_variable"])
assert providers["gemini"]["key_variable"] == "GOOGLE_API_KEY"

**Check:** No. The configured endpoint/client and model select the service. Local Ollama still needs its service and downloaded model.

**Change one thing:** Add a provider from the linked source. Copy its endpoint and variable name, never a credential value.

## 3. Configure a real SDK agent and inspect its tool schema

Agent holds instructions/model/tools. The function_tool decorator constructs a schema from the signature and docstring. Constructing this configuration does not run the agent or contact a provider.

**Predict:** Which argument names will appear in the registered tool schema?

In [ ]:
from agents import Agent, function_tool, set_tracing_disabled
set_tracing_disabled(True)
@function_tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b
agent = Agent(name="Practice teacher", instructions="Use add_numbers for addition.",
              model="gpt-4o-mini", tools=[add_numbers])
print(agent.name, agent.tools[0].name)
print(sorted(agent.tools[0].params_json_schema["properties"]))
assert sorted(agent.tools[0].params_json_schema["properties"]) == ["a", "b"]

**Check:** a and b. The schema advertises the function; Runner executes the model/tool workflow when deliberately invoked.

**Change one thing:** Change the docstring and inspect the tool description. Find Runner.run in the source: that is where the interaction loop actually begins.

## 4. Client, request, and response are separate objects

Constructing a client prepares a connection interface. A request invokes it. The returned response contains choices, a message, and its content. This fixture imitates that nested shape.

**Predict:** What does choices[0] select?

In [ ]:
from types import SimpleNamespace
response = SimpleNamespace(choices=[
    SimpleNamespace(message=SimpleNamespace(content="The answer is 4."))
])
answer = response.choices[0].message.content
print(answer)
assert answer == "The answer is 4."

**Check:** The first returned choice. This is simulated response data, not a model call.

**Change one thing:** Add a second choice with different text. Read choices[1] and explain why choosing an index matters.

## 5. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.